# 4.1 Distributions — a hypothesis about the process

A distribution is a hypothesis about the process that generated the data.

"Lognormal with μ=3.2, σ=0.8" is a description you can check against what you know about
how the numbers were produced: a lognormal says the value is a *product* of many factors,
which is what gives it the long tail. "Mean 34, std 41" describes nothing, because for a
lognormal the mean is not typical and the std implies a symmetry that is not there.

Three reasons to fit a distribution rather than report a mean, in increasing order of value:

1. **To summarise honestly.** Two parameters that describe the shape, instead of two that
   assume one.
2. **To decide what is unusual.** A point is only an outlier *relative to a distribution*.
   "Three standard deviations from the mean" is a statement about the normal distribution
   you did not know you had assumed.
3. **To compare two situations.** If the same family fits before and after an event, the
   parameters say *what* changed — the rate, the spread, the scale — and not just *that*
   something did. That is the move this lesson builds towards.

This notebook answers the first question — *which shapes are there, and what produces
each* — on generated data, so that nothing has to be estimated. The lesson then takes the
families to real data in four steps:

- [04.2](04.2-long-tail.ipynb) — one family on one population: the long tail of taxi fares.
- [04.3](04.3-penguins.ipynb) — a fit that fails because the population is several groups.
- [04.4](04.4-before-and-after.ipynb) — the same family on both sides of an event, and a
  null distribution to say how sure.
- [04.5](04.5-fitting-the-residual.ipynb) — a distribution fitted to what a model left over.

## What goad gives you for this

Lesson 1 gave data decisions a home in a `Pipeline`, lesson 2 gave plots one in `BasePlot`.
Distribution fitting gets the same treatment, in three pieces:

- **`DistributionRegistry`** holds the families goad will try — `norm`, `lognorm`,
  `exponential`, `gamma`, `weibull`, `poisson`, `nbinom`, and a few more — each with the
  metadata a fit needs: the scipy object, whether it is discrete, how many parameters. You
  can register a family it does not ship (`registry.register_distribution("pareto",
  stats.pareto, is_discrete=False, num_params=3)`), and every registry starts from the
  shipped set, so a registration in one cell does not leak into the next.
- **`DistributionFitter(registry, seed=42).fit(data, discrete=...)`** fits every family
  of the right kind by maximum likelihood and returns one result per family — a `FitResult` with
  the parameters, a `frozen_dist` you can draw or sample from, the log-likelihood, and a
  Kolmogorov–Smirnov test; or a `FailedFit` carrying the reason. Failures are values, so
  one badly-behaved family does not lose you the comparison. `discrete` is *your* call:
  counts are discrete, durations are not, and getting it wrong does not error — it fits
  the wrong half of the registry. `fit_distribution("lognorm", data)` fits one family
  when you already know which. The `seed` is there because the optimizer behind the fit
  is stochastic: pass one when the numbers must repeat, as they must in a lesson; leave it
  out in your own work and a fit that moves between runs is telling you something.
- **`fit_table(results)`** turns those results into the ranked table you hand in: family,
  parameters, log-likelihood, KS statistic and p-value, and which criteria it won.

Two winners are marked, on purpose. **Log-likelihood** rewards a family that puts high
probability on the points you observed — it is dominated by the bulk of the data. **KS**
measures the largest gap between the fitted and the empirical cumulative distribution — it
is sensitive to the middle and comparatively blind in the tails. They usually agree; when
they do not, that disagreement is the most informative thing on the table, and the answer
is a picture, not an average of the two numbers.

The pictures: `PlotFits` (the histogram with the top fits over it), `QQPlot` (sorted data
against a fitted family's quantiles — the tails, where families differ and histograms are
unreadable), `ECDFPlot` (two samples on one bin-free axis), and `DistPlot` (draw a family
on its own, or over a histogram). All of them are `BasePlot`s: `create_figure(n_plots=...)`
and `plot_on_axes(...)` compose them the way every plot in this course composes.

In [ ]:
import numpy as np
from scipy import stats

from goad_toolkit.visualizer import DistPlot, HistogramPlot, PlotSettings

rng = np.random.default_rng(42)

## 4.1.1 Six families, generated — before any real data is involved

Each family below is sampled from a known generator and drawn with its own curve over the
sample, so what you see is *only* the shape: no fitting, no "did we estimate the parameters
right". The one-liner under each name is the mechanism that produces it — the thing to
recognise in your own data before you fit anything.

In [ ]:
families = [
    ("norm", stats.norm(loc=0, scale=1), "sums of many independent contributions"),
    ("lognorm", stats.lognorm(s=0.8, scale=np.exp(1)), "products of many factors"),
    ("expon", stats.expon(scale=1), "time between events arriving at a steady rate"),
    ("poisson", stats.poisson(mu=4), "counts of events in a fixed window, steady rate"),
    ("weibull", stats.weibull_min(c=1.5, scale=1), "time-to-failure with a changing hazard"),
    ("pareto", stats.pareto(b=2.5), "preferential attachment, rich-get-richer"),
]

gallery = PlotSettings(
    figsize=(13, 7),
    title="Six families, generated",
    subplot_titles=[f"{name}\n{mechanism}" for name, _, mechanism in families],
    max_cols=3,
)
host = HistogramPlot(gallery)
fig, axes = host.create_figure(n_plots=len(families))
for ax, (name, dist, mechanism) in zip(axes, families):
    sample = dist.rvs(2000, random_state=rng)
    # discrete=True gives a count one bar per integer, so the bars and the pmf share a scale
    host.plot_on_axes(HistogramPlot(gallery), ax, data=sample, color="lightgrey",
                      discrete=(name == "poisson"))
    host.plot_on_axes(DistPlot(gallery), ax, distribution=dist, color="crimson")
    ax.set_ylabel("")

Two of these are counts, not measurements: `poisson` has mass only at the integers, so its
curve is drawn through them — a discrete count has no density, only a probability per value.
And two of them are hard to tell apart from a histogram: `lognorm` and `pareto` both just
look "long-tailed". The discriminator is the rank–frequency plot on log-log axes (a power law
is straight there, a lognormal curves) — goad's Distributions chapter, §5.7, if you ever
need it.

## 4.1.2 The central limit theorem — sums converge, nothing else has to

Sums of *anything* with finite variance drift toward normal as you add more terms. That is
why normal turns up wherever sums turn up — and it says nothing at all about products,
waiting times, or counts, which converge to something else entirely (lognormal, gamma,
Poisson) precisely because they are not sums.

In [ ]:
ns = [1, 2, 5, 30]
clt = PlotSettings(
    figsize=(14, 3.2),  # ty: ignore[invalid-argument-type]
    title="Sums of uniforms converge to normal as n grows",
    subplot_titles=[f"sum of {n} uniform draw{'s' if n > 1 else ''}" for n in ns],
    max_cols=4,
    sharex=True,
    sharey=True,
)
host = HistogramPlot(clt)
fig, axes = host.create_figure(n_plots=len(ns))
for ax, n in zip(axes, ns):
    sums = stats.uniform(loc=-1, scale=2).rvs((5000, n), random_state=rng).sum(axis=1)
    sums = (sums - sums.mean()) / sums.std()
    host.plot_on_axes(HistogramPlot(clt), ax, data=sums, color="steelblue")
    host.plot_on_axes(DistPlot(clt), ax, distribution=stats.norm(0, 1), x_range=(-4, 4))
    ax.set_ylabel("")

In [ ]:
# A PRODUCT of the same uniforms does not converge to normal at all: it converges to
# lognormal, because a sum of logs is a sum, and exp() of a sum is a product.
n = 30
sums = stats.uniform(loc=1, scale=1).rvs((5000, n), random_state=rng).sum(axis=1)
products = stats.uniform(loc=1, scale=1).rvs((5000, n), random_state=rng).prod(axis=1)

product = PlotSettings(
    figsize=(12, 3.2),  # ty: ignore[invalid-argument-type]
    title="The CLT is a statement about sums",
    subplot_titles=["sum → normal", "product → long tail", "log(product) → normal again"],
    max_cols=3,
)
host = HistogramPlot(product)
fig, axes = host.create_figure(n_plots=3)
for ax, values in zip(axes, [sums, products, np.log(products)]):
    host.plot_on_axes(HistogramPlot(product), ax, data=values, color="steelblue")
    ax.set_ylabel("")

The right panel is `log(product)`, not `product` — that is the whole trick lognormal is
named for. Plot `products` directly and it is skewed and long-tailed, not bell-shaped,
because a product of positive numbers is exactly the kind of variable the CLT does not
apply to directly. Take the log and it becomes a sum, and the theorem applies again.

---

**Where this goes next.** Every curve above was drawn from parameters you chose. The next
step is the reverse: real numbers, an unknown family, and a fitter that has to choose one and
say how well it did. [04.2-long-tail](04.2-long-tail.ipynb) does that on taxi fares, where the
mechanism predicts a lognormal and the fitter gets to agree or not.